In [ ]:
! pip install cebra 

In [1]:
# code cell to make a dataset and labels for further training 

from pathlib import Path
import mne
import numpy as np
import random

# ----------------------------------------
# Input/output paths
# ----------------------------------------
#for Michelle: change these paths according to how you upload your files (you can upload the folders
# with the same names and then not change anything here) 
RAW_DIRS = {
    "nt9-10": Path("/kaggle/input/ica-processed-nt9-10"),
    "nt34-35": Path("/kaggle/input/nt34-35"),
    "nt21-22": Path("/kaggle/input/nt21-22/Preprocessed_nt21-22/cut"),
    "nt38-39": Path("/kaggle/input/nt38-39/Preprocessed_nt38-39"), 
    "nt40-41": Path("/kaggle/input/nt40-41/Preprocessed_nt40-41"), 
    "nt13-14": Path("/kaggle/input/nt13-14/Preprocessed_nt13-14/cut"),
    "nt36-37": Path("/kaggle/input/nt36-37/Preprocessed_nt36-37/cut"), 
    "nt11-12": Path("/kaggle/input/nt11-12/Preprocessed_nt11-12"), 
    "nt15-16": Path("/kaggle/input/nt15-16/Preprocessed_nt15-16"), 
    "nt19-20": Path("/kaggle/input/nt19-20/Preprocessed_nt19-20"), 
    "nt23-24": Path("/kaggle/input/nt23-24/Preprocessed_nt23-24"), 
    "nt27-28": Path("/kaggle/input/nt27-28/Preprocessed_nt27-28a"), 
    "nt32-33": Path("/kaggle/input/nt32-33/Preprocessed_nt32-33"), 
    "nt43-44": Path("/kaggle/input/nt43-44")
}

OUTPUT_DIR = Path("/kaggle/working/cebra_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ----------------------------------------
# Pairings: conversation recordings 
# ----------------------------------------

#for Michelle: normally, you don't have to change the filenames, as they correspond to the ones in Box
# you should change the last number which is gender code 

PAIRINGS = {
    "first":   ("nt9-10",  "nt9_listen_cut_60_components_preprocessed.edf",  "nt10_speak_cut_60_components_preprocessed.edf",1),
    "second":  ("nt9-10",  "nt10_listen_cut_60_components_preprocessed.edf", "nt9_speak_cut_60_components_preprocessed.edf",1),
    "third":   ("nt34-35", "nt34_listen_cut_preprocessed.edf",              "nt35_speak_cut_preprocessed.edf",1),
    "fourth":  ("nt34-35", "nt35_listen_cut_preprocessed.edf",              "nt34_speak_cut_preprocessed.edf", 1),
    "fifth":   ("nt21-22", "nt21_listening_preprocessed.edf",              "nt22_speaking_preprocessed.edf",1), 
    "sixth":   ("nt21-22", "nt22_listening_preprocessed.edf",              "nt21_speaking_preprocessed.edf",1),
    "seventh": ("nt38-39", "nt38_listen_preprocessed.edf",                 "nt39_speak_preprocessed.edf",1), 
    "eighth":  ("nt38-39", "nt39_listen_preprocessed.edf",                 "nt38_speak_preprocessed.edf",1),
    "ninth":   ("nt40-41", "nt40_listen_preprocessed.edf",                 "nt41_speak_preprocessed.edf",1), 
    "tenth":   ("nt40-41", "nt41_listen_preprocessed.edf",                 "nt40_speak_preprocessed.edf",1),
    "eleventh":   ("nt36-37", "nt36_listening_preprocessed.edf",              "nt37_speaking_preprocessed.edf",1), 
    "twelveth":   ("nt36-37", "nt37_listening_preprocessed.edf",              "nt36_speaking_preprocessed.edf",1),
    "thirteenth":   ("nt13-14", "nt13_listening_preprocessed.edf",              "nt14_speaking_preprocessed.edf",1), 
    "fourteenth":   ("nt13-14", "nt14_listening_preprocessed.edf",              "nt13_speaking_preprocessed.edf",1), 
    "fifteenth": ("nt11-12", "nt11_listening_preprocessed.edf", "nt12_speaking_preprocessed.edf",1), 
    "sixteenth": ("nt11-12", "nt12_listening_preprocessed.edf", "nt11_speaking_preprocessed.edf",1), 
    "seventeenth": ("nt15-16", "nt15_listen_preprocessed.edf", "nt16_speak_preprocessed.edf",1), 
    "eighteenth": ("nt15-16", "nt16_listen_preprocessed.edf", "nt15_speak_preprocessed.edf", 1), 
    "nineteenth": ("nt19-20", "nt19_listening_preprocessed.edf", "nt20_speaking_preprocessed.edf",1), 
    "twentieth": ("nt19-20", "nt20_listening_preprocessed.edf", "nt19_speaking_preprocessed.edf",1), 
    "twentyfirst": ("nt23-24", "nt23_listen_preprocessed.edf", "nt24_speak_preprocessed.edf",1), 
    "twentysecond": ("nt23-24", "nt24_listen_preprocessed.edf", "nt23_speak_preprocessed.edf",1), 
    "twentythird": ("nt27-28", "nt27_listening_preprocessed.edf", "nt28a_speaking_preprocessed.edf",1), 
    "twentyfourth": ("nt27-28", "nt28a_listening_preprocessed.edf", "nt27_speaking_preprocessed.edf",1),
    "twentyfifth": ("nt32-33", "nt32_listen_preprocessed.edf", "nt33_speak_preprocessed.edf",1), 
    "twentysixth": ("nt32-33", "nt33_listen_preprocessed.edf", "nt32_speak_preprocessed.edf",1),
    "twentyseventh": ("nt43-44", "nt43_listen_preprocessed.edf", "nt44_speak_preprocessed.edf",1),
    "twentyeighth": ("nt43-44", "nt44_listen_preprocessed.edf", "nt43_speak_preprocessed.edf",1), 
    
}

# ----------------------------------------
# Helper functions
# ----------------------------------------
def load_eeg_raw(path):
    return mne.io.read_raw_edf(path, preload=True, verbose=False)

def align_lengths(a, b):
    T = min(a.shape[1], b.shape[1])
    return a[:, :T], b[:, :T]

def minmax_per_channel(x):
    xmin = x.min(axis=1, keepdims=True)
    xmax = x.max(axis=1, keepdims=True)
    rng = np.where((xmax - xmin) == 0, 1, xmax - xmin)
    return (x - xmin) / rng

# ----------------------------------------
# Data containers
# ----------------------------------------
all_data = []
labels_v8 = []

# ----------------------------------------
# Main loop for creating the dataset 
# ----------------------------------------
for idx, (pair_name, (source_key, file1, file2, gender_num)) in enumerate(PAIRINGS.items()):
    source_dir = RAW_DIRS[source_key]
    raw1 = load_eeg_raw(source_dir / file1)
    raw2 = load_eeg_raw(source_dir / file2)
    sfreq1, sfreq2 = raw1.info['sfreq'], raw2.info['sfreq']

    montage_file = Path("/kaggle/input/montage/GSN-HydroCel-65_1.0.sfp")

    # Get clean EEG data
    A = raw1.get_data(picks="eeg")
    B = raw2.get_data(picks="eeg")
    A, B = align_lengths(A, B)
    stacked = np.vstack([A, B])
    normalized = minmax_per_channel(stacked)
    np.save(OUTPUT_DIR / f"{pair_name}_normalized.npy", normalized)
    print(f"✓ Saved: {pair_name}_normalized.npy  {normalized.shape}")
    all_data.append(normalized)

    # Labels
    T = normalized.shape[1]
    v8_labels = gender_num 
    labels_v8.append(np.full(T, v8_label, dtype=np.int32))


combined_data = np.hstack(all_data)
combined_labels_v8 = np.hstack(labels_v8)

np.save(OUTPUT_DIR / "combined_normalized.npy", combined_data)
np.save(OUTPUT_DIR / "combined_labels_v8.npy", combined_labels_v8)

print(f"✓ Saved combined_normalized.npy             {combined_data.shape}")
print(f"✓ Saved combined_labels_v8.npy              {combined_labels_v8.shape}")

In [ ]:
from cebra import CEBRA, plot_temperature, plot_embedding, plot_loss, KNNDecoder
from cebra.integrations.sklearn import metrics as cmetrics
from matplotlib import colors as mcolors, cm as mcm
import torch
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
import matplotlib.patches as mpatches

# Output directory
OUTPUT_DIR = Path("/kaggle/working/cebra_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ===============================
# Balanced 80/20 train/test split
# ===============================
def train_test_split_all_blocks(embeddings, labels, seed=0, test_size=0.2, block_length=None):
    """
    Split embeddings and labels into train/test sets with stratification by label.

    embeddings: np.ndarray, shape (T, features)
    labels: np.ndarray, shape (T,)
    seed: random seed
    test_size: fraction of samples in the test set
    block_length: int or None, if not None, split into blocks of this length before sampling
    """
    rng = np.random.default_rng(seed)
    unique_labels = np.unique(labels)

    train_X, test_X, train_y, test_y = [], [], [], []

    for label in unique_labels:
        label_idx = np.where(labels == label)[0]

        if block_length is not None:
            # break into full blocks of equal length
            blocks = [label_idx[i:i+block_length]
                      for i in range(0, len(label_idx), block_length)
                      if len(label_idx[i:i+block_length]) == block_length]
            rng.shuffle(blocks)
            n_test_blocks = int(len(blocks) * test_size)
            test_blocks = blocks[:n_test_blocks]
            train_blocks = blocks[n_test_blocks:]

            for block in train_blocks:
                train_X.append(embeddings[block])
                train_y.append(labels[block])
            for block in test_blocks:
                test_X.append(embeddings[block])
                test_y.append(labels[block])

        else:
            # random individual sampling within this label
            rng.shuffle(label_idx)
            n_test = int(len(label_idx) * test_size)
            test_idx = label_idx[:n_test]
            train_idx = label_idx[n_test:]

            train_X.append(embeddings[train_idx])
            train_y.append(labels[train_idx])
            test_X.append(embeddings[test_idx])
            test_y.append(labels[test_idx])

    # Combine across all labels
    return (
        np.concatenate(train_X, axis=0),
        np.concatenate(test_X, axis=0),
        np.concatenate(train_y, axis=0),
        np.concatenate(test_y, axis=0),
    )

MODEL_KWARGS = dict(
    model_architecture="offset10-model",
    batch_size=512,
    learning_rate=3e-4,
    temperature_mode = 'auto',
    temperature=1,
    min_temperature = 1e-1,
    max_iterations=5000,
    conditional="time_delta",
    output_dimension=3,
    distance="cosine",
    device="cuda:0",
    verbose=True,
    time_offsets=10,
)

# =====================
# Function for one CEBRA session
# =====================
def run_cebra_combined(run_id, label_version="v8", seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.manual_seed(run_id)
    model = CEBRA(**MODEL_KWARGS)

     # Load base training data (always clean)
    X_train = np.load(OUTPUT_DIR / "combined_normalized.npy").T
    Y_train = np.load(OUTPUT_DIR / f"combined_labels_{label_version}.npy")

    # Train 
    model.fit(X_train, Y_train)

    # Compute embeddings 
    emb = model.transform(X_eval)

    # Save artifacts
    np.save(OUTPUT_DIR / f"{tag}_emb.npy", emb)
    model.save(OUTPUT_DIR / f"{tag}_model.pt")

    # make embedding labels 
    colors = plt.cm.tab20b.colors
    plot_embedding(emb, Y_eval, cmap="tab20b")
        # Call plot_embedding as usual
        # Y_eval should be a 1D numeric array (int or float)
    labels = np.asarray(Y_eval).ravel().astype(float)
    classes = np.unique(labels)
    
    cmap = plt.get_cmap("tab20b")      # keep whatever cmap you pass to plot_embedding
    norm = mcolors.Normalize(vmin=labels.min(), vmax=labels.max())  # match scatter's default norm
    
    # Plot with the SAME cmap+norm and disable 3D depth shading
    ax = plot_embedding(emb, labels, cmap=cmap, norm=norm, depthshade=False)
    
    # Build legend colors using the SAME mapping
    sm = mcm.ScalarMappable(norm=norm, cmap=cmap)
    handles = [mpatches.Patch(color=sm.to_rgba(lbl), label=str(int(lbl) if lbl.is_integer() else str(lbl)))
               for lbl in classes]
    
    ax.legend(handles=handles, title="Label", loc="best", frameon=True)
    plt.savefig(OUTPUT_DIR / f"{tag}_embedding.png")
    plt.close()

    plot_loss(model)
    plt.savefig(OUTPUT_DIR / f"{tag}_loss.png")
    plt.close()

    plot_temperature(model)
    plt.savefig(OUTPUT_DIR / f"{tag}_temperature.png")
    plt.close()

     # Metrics
    gof = cmetrics.goodness_of_fit_score(model, X_eval, Y_eval)

    # Classification of embedding 
    train_X, test_X, train_y, test_y = train_test_split_all_blocks(emb, Y_eval, seed=seed)
    knn = KNNDecoder(n_neighbors=5)
    knn.fit(train_X, train_y)
    acc = knn.score(test_X, test_y)

    result = {
        "run_id": run_id,
        "label_version": label_version,
        "goodness_of_fit": gof,
        "knn_acc": acc,
        "learned_temperature": float(min(model.state_dict_['log']['temperature'])), 
        "seed": seed, 
    }

    with open(OUTPUT_DIR / f"{tag}_metrics.json", "w") as f:
        json.dump(result, f, indent=2)

    print(f"✓ Run {run_id} [{label_version}] "
          f"GOF={gof:.4f}, KNN={acc:.4f}")


# ===============
# Run the code 
# ===============
RUNS = list(range(5)) # you can change the number of runs here 

for run_id in RUNS:
        run_cebra_combined(run_id)